In [16]:
# Run locally: no Google Drive mount needed
# Make sure your datasets are in the ../Datasets/ folder


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
dataset='../Datasets/dataset_web_eradah.csv'

# Web Search Dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, GRU, Input, LayerNormalization
from tensorflow.keras.optimizers import Adam
import math
from scipy.stats import pearsonr

# Function to calculate MAPE
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Function to calculate sMAPE
def symmetric_mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))



# Read the dataset
df = pd.read_csv(dataset)

# Convert all columns to numeric
for col in df.columns[1:]:
    df[col] = pd.to_numeric(df[col])

# Convert month to datetime and set as index
df['month'] = pd.to_datetime(df['month'])
df.set_index('month', inplace=True)


# Display dataset info
print("Dataset shape:", df.shape)
print("\nDataset info:")
print(df.info())
print("\nFirst few rows:")
print(df.head())

# Separate features and target
feature_columns = ['depression', 'tired', 'weight loss', 'anxiety', 'sad']
target_column = ['actual']

X = df[feature_columns]
y = df[target_column]

# Display original ranges
print("\nOriginal feature ranges:")
print(X.agg(['min', 'max']))
print("\nOriginal target range:")
print(y.agg(['min', 'max']))

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

# Scale features and target separately
X_scaled = feature_scaler.fit_transform(X)
y_scaled = target_scaler.fit_transform(y)

# Create scaled DataFrames
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_columns, index=df.index)
y_scaled_df = pd.DataFrame(y_scaled, columns=target_column, index=df.index)

# Combine
scaled_df = pd.concat([X_scaled_df, y_scaled_df], axis=1)

print("\nAfter scaling - Feature ranges (0-1):")
print(X_scaled_df.agg(['min', 'max']))
print("\nAfter scaling - Target range (0-1):")
print(y_scaled_df.agg(['min', 'max']))


scaled_data = np.column_stack([X_scaled, y_scaled])


# Create sequences
def create_sequences(data, time_steps):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:(i + time_steps), :-1])  # All features except the target
        y.append(data[i + time_steps, -1])       # Target (actual cases)
    return np.array(X), np.array(y)

# Set time steps and create sequences
time_steps = 12
X, y = create_sequences(scaled_data, time_steps)

print(f"\nShape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

# Split the data: 55% training, 20% validation, 25% test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.25, random_state=42, shuffle=False)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.267, random_state=42, shuffle=False)

print(f"Training set: {X_train.shape}, {y_train.shape}")
print(f"Validation set: {X_val.shape}, {y_val.shape}")
print(f"Test set: {X_test.shape}, {y_test.shape}")

# Build GRU model
model = Sequential([
    Input(shape=(X_train.shape[1], X_train.shape[2])),
    Bidirectional(GRU(128, return_sequences=True)),
    LayerNormalization(),
    Bidirectional(GRU(64, return_sequences=False)),
    LayerNormalization(),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1)
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=40, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("best_model.keras", save_best_only=True)
]
# Display model summary
print("\nModel Summary:")
model.summary()

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=8,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1,
    shuffle=False
)

# Make predictions
y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

# Calculate evaluation metrics
def calculate_metrics(y_true, y_pred, set_name):
    mse = mean_squared_error(y_true, y_pred)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    smape = symmetric_mean_absolute_percentage_error(y_true, y_pred)

    print(f"\n{set_name} Set Metrics:")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MAPE: {mape:.2f}%")
    print(f"sMAPE: {smape:.2f}%")

    return mse, rmse, mae, mape, smape

# Calculate metrics for all sets
train_metrics = calculate_metrics(y_train, y_pred_train, "Training")
val_metrics = calculate_metrics(y_val, y_pred_val, "Validation")
test_metrics = calculate_metrics(y_test, y_pred_test, "Test")

# Create a comparison DataFrame
metrics_df = pd.DataFrame({
    'Dataset': ['Training', 'Validation', 'Test'],
    'MSE': [train_metrics[0], val_metrics[0], test_metrics[0]],
    'RMSE': [train_metrics[1], val_metrics[1], test_metrics[1]],
    'MAE': [train_metrics[2], val_metrics[2], test_metrics[2]],
    'sMAPE': [train_metrics[4], val_metrics[4], test_metrics[4]]
})

print("\n" + "="*50)
print("COMPARISON OF ALL DATASET PERFORMANCE")
print("="*50)
print(metrics_df.round(4))

# Plot training history
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss During Training')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()



all_months = df.index[time_steps:]

train_months = all_months[:len(y_train)]
val_months = all_months[len(y_train):len(y_train) + len(y_val)]
test_months = all_months[-len(y_test):]

# Ensure 1-D arrays
y_train= np.array(y_train).reshape(-1)
y_pred_train = np.array(y_pred_train).reshape(-1)

y_val = np.array(y_val).reshape(-1)
y_pred_val = np.array(y_pred_val).reshape(-1)

y_test = np.array(y_test).reshape(-1)
y_pred_test = np.array(y_pred_test).reshape(-1)

# --- Train ---
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title("Train Phase: Actual vs Predicted Depression Cases")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")
ax1.plot(train_months, y_train, label="Train Actual", color="orange", marker="o", alpha=0.6)
ax1.tick_params(axis='y', labelcolor="black")
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(train_months, y_pred_train, label="Train Predicted", color="red", linestyle="--", marker="x", alpha=0.9)
ax2.tick_params(axis='y', labelcolor="black")

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left")

plt.show()

# --- Validation ---
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title("Validation Phase: Actual vs Predicted Depression Cases")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")
ax1.plot(val_months, y_val, label="Validation Actual", color="pink", marker="o", alpha=0.6)
ax1.tick_params(axis='y', labelcolor="black")
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(val_months, y_pred_val, label="Validation Predicted", color="purple", linestyle="--", marker="x", alpha=0.9)
ax2.tick_params(axis='y', labelcolor="black")

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left")

plt.show()

# --- Test ---
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title("Test Phase: Actual vs Predicted Depression Cases")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")
ax1.plot(test_months, y_test, label="Test Actual", color="lightblue", marker="o", alpha=0.6)
ax1.tick_params(axis='y', labelcolor="black")
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(test_months, y_pred_test, label="Test Predicted", color="blue", linestyle="--", marker="x", alpha=0.9)
ax2.tick_params(axis='y', labelcolor="black")

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left")

plt.show()

# ==== Anomaly Detection ====

# Convert to binary (1 = anomaly, 0 = normal)
def convert_to_binary(values, threshold):
    return (values > threshold).astype(int)

k_values = [0.5, 0.7, 0.9, 1, 1.3, 1.5, 1.7, 2]

# --- Train ---
results_train = []
for k_actual in k_values:
  threshold_train_actual = (y_train.mean() + (k_actual * y_train.std()))
  for k_pred in k_values:
    threshold_train_pred = (y_pred_train.mean() + (k_pred * y_pred_train.std()))

    y_train_binary = convert_to_binary(y_train, threshold_train_actual)
    y_pred_train_binary = convert_to_binary(y_pred_train, threshold_train_pred)

    binary_df = pd.DataFrame({
        'Actual': y_train_binary.flatten(),
        'Predicted': y_pred_train_binary.flatten()
    })

    # Save to CSV file
    '''filename = f'../outputs/GRU/anomaly_detection_train_kActual_{k_actual:.1f}_kPred_{k_pred:.1f}_web_search_GRU.csv'
    binary_df.to_csv(filename, index=False)
    print(f"Saved binary data to: {filename}")'''

    accuracy = accuracy_score(y_train_binary, y_pred_train_binary)
    precision = precision_score(y_train_binary, y_pred_train_binary, zero_division=0)
    recall = recall_score(y_train_binary, y_pred_train_binary, zero_division=0)
    f1 = f1_score(y_train_binary, y_pred_train_binary, zero_division=0)

    results_train.append({
        'K_actual': k_actual,
        'K_pred': k_pred,
        'threshold_train_pred': threshold_train_pred,
        'threshold_train_actual': threshold_train_actual,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    })

results_df_train = pd.DataFrame(results_train)
print("\n" + "="*60)
print("SUMMARY OF ALL K VALUES - TRAINING")
print("="*60)
print(results_df_train.round(4))

# Best k
best_f1_idx_train = results_df_train['F1 Score'].idxmax()
best_row_train = results_df_train.loc[best_f1_idx_train]

print("\n" + "="*60)
print("BEST K BASED ON F1 SCORE - TRAINING")
print("="*60)
print(best_row_train)

# Apply best k
y_train_binary_actual = convert_to_binary(y_train, best_row_train['threshold_train_actual'])
y_train_binary_pred = convert_to_binary(y_pred_train, best_row_train['threshold_train_pred'])

train_months = np.array(train_months)
y_pred_train = y_pred_train.flatten()
# Plot anomalies
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title(f"Train Phase: Anomaly Detection (k_actual={best_row_train['K_actual']}, k_pred={best_row_train['K_pred']})")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")
ax1.plot(train_months, y_train, label="Train Actual", color="orange", marker="o", alpha=0.6)
ax1.scatter(train_months[y_train_binary_actual == 1], y_train[y_train_binary_actual == 1],
            color="black", marker="x", s=100, label="Actual Anomaly")

# Add threshold line for actual values
ax1.axhline(best_row_train['threshold_train_actual'], color='orange', linestyle='--', label='Actual Threshold')

ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(train_months, y_pred_train, label="Train Predicted", color="red", linestyle="--", marker="x", alpha=0.9)
ax2.scatter(train_months[y_train_binary_pred == 1], y_pred_train[y_train_binary_pred == 1],
            color="blue", marker="x", s=100, label="Predicted Anomaly")

# Add threshold line for predicted values
ax2.axhline(best_row_train['threshold_train_pred'], color='red', linestyle='--', label='Predicted Threshold')

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

plt.show()

# Confusion matrix
cm_train = confusion_matrix(y_train_binary_actual, y_train_binary_pred)
print("\nConfusion Matrix (Training Set):")
print(cm_train)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_train, display_labels=["Normal", "Anomaly"])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix - Training Set (Best k_actual={best_row_train['K_actual']}, k_pred={best_row_train['K_pred']})")
plt.show()

print("=== AFTER APPLYING SHIFT ===")

shift_results = []
shift_range = np.arange(1, 4)

for s in shift_range:
    # Apply shift
    shifted_predict_train = np.roll(y_pred_train, s)

    for k_actual in k_values:
      threshold_train_actual = (y_train.mean() + (k_actual * y_train.std()))
      for k_pred in k_values:
        threshold_train_pred = (shifted_predict_train.mean() + (k_pred * shifted_predict_train.std()))

        y_train_binary = convert_to_binary(y_train, threshold_train_actual)
        y_pred_train_binary = convert_to_binary(shifted_predict_train, threshold_train_pred)

        binary_df = pd.DataFrame({
            'Actual': y_train_binary.flatten(),
            'Predicted': y_pred_train_binary.flatten()
        })
        '''filename = f'../outputs/GRU/anomaly_detection_train_shift_{s}_kActual_{k_actual:.1f}_kPred_{k_pred:.1f}_web_search_GRU.csv'
        binary_df.to_csv(filename, index=False)
        print(f"Saved binary data to: {filename}")'''

        accuracy = accuracy_score(y_train_binary, y_pred_train_binary)
        precision = precision_score(y_train_binary, y_pred_train_binary, zero_division=0)
        recall = recall_score(y_train_binary, y_pred_train_binary, zero_division=0)
        f1 = f1_score(y_train_binary, y_pred_train_binary, zero_division=0)

        shift_results.append({
            'Shift': s,
            'K_actual': k_actual,
            'K_pred': k_pred,
            'threshold_train_pred': threshold_train_pred,
            'threshold_train_actual': threshold_train_actual,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1
        })

shift_results_df = pd.DataFrame(shift_results)
print("\n" + "="*60)
print("SUMMARY OF ALL SHIFT VALUES AND K VALUES - TRAINING")
print("="*60)
print(shift_results_df.round(4))

best_f1_idx_shift_train = shift_results_df['F1 Score'].idxmax()
best_row_shift_train = shift_results_df.loc[best_f1_idx_shift_train]

print("\n" + "="*60)
print("BEST SHIFT BASED ON F1 SCORE - TRAINING")
print("="*60)
print(best_row_shift_train)

# === PLOT & CONFUSION MATRIX FOR BEST SHIFT AND K ===

best_shift = int(best_row_shift_train['Shift'])
best_k_actual = best_row_shift_train['K_actual']
best_k_pred = best_row_shift_train['K_pred']

# Apply shift
shifted_predict_train = np.roll(y_pred_train, best_shift)

threshold_train_pred = shifted_predict_train.mean() + (best_k_pred * shifted_predict_train.std())
threshold_train_actual = y_train.mean() + (best_k_actual * y_train.std())

y_train_binary_actual = convert_to_binary(y_train, threshold_train_actual)
y_train_binary_pred = convert_to_binary(shifted_predict_train, threshold_train_pred)

# --- PLOT ---
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title(f"Train Phase (After Shift): Anomaly Detection (Shift={best_shift}, k_actual={best_k_actual}, k_pred={best_k_pred})")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")

ax1.plot(train_months, y_train, label="Train Actual", color="orange", marker="o", alpha=0.6)
ax1.scatter(train_months[y_train_binary_actual == 1], y_train[y_train_binary_actual == 1],
            color="black", marker="x", s=100, label="Actual Anomaly")
ax1.axhline(threshold_train_actual, color='orange', linestyle='--', label='Actual Threshold')
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(train_months, shifted_predict_train, label="Train Predicted (Shifted)", color="red", linestyle="--", marker="x", alpha=0.9)
ax2.scatter(train_months[y_train_binary_pred == 1], shifted_predict_train[y_train_binary_pred == 1],
            color="blue", marker="x", s=100, label="Predicted Anomaly")
ax2.axhline(threshold_train_pred, color='red', linestyle='--', label='Predicted Threshold')

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

plt.show()

# --- CONFUSION MATRIX ---
cm_shift_train = confusion_matrix(y_train_binary_actual, y_train_binary_pred)
print("\nConfusion Matrix (Training Set):")
print(cm_shift_train)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_shift_train, display_labels=["Normal", "Anomaly"])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix - Training Set (Shift={best_shift}, k_actual={best_k_actual}, k_pred={best_k_pred})")
plt.show()

# Compare with original (no shift) results
print("\n" + "="*60)
print("COMPARISON: NO SHIFT vs BEST SHIFT")
print("="*60)
print(f"No Shift - Best K_actual: {best_row_train['K_actual']}, K_pred: {best_row_train['K_pred']}, F1 Score: {best_row_train['F1 Score']:.4f}")
print(f"With Shift - Best Shift: {best_shift}, Best K_actual: {best_k_actual}, K_pred: {best_k_pred}, F1 Score: {best_row_shift_train['F1 Score']:.4f}")
print(f"F1 Score Improvement: {best_row_shift_train['F1 Score'] - best_row_train['F1 Score']:.4f}")

results_val = []

for k in k_values:
    threshold_val_pred = (y_pred_val.mean() + (k * y_pred_val.std()))
    threshold_val_actual = (y_val.mean() + (k * y_val.std()))

    y_val_binary = convert_to_binary(y_val, threshold_val_actual)
    y_pred_val_binary = convert_to_binary(y_pred_val, threshold_val_pred)

    binary_df = pd.DataFrame({
        'Actual': y_val_binary.flatten(),
        'Predicted': y_pred_val_binary.flatten()
    })
    '''filename = f'../outputs/GRU/anomaly_detection_val_k_{k:.1f}_web_search_GRU.csv'
    binary_df.to_csv(filename, index=False)
    print(f"Saved binary data to: {filename}")'''

    accuracy = accuracy_score(y_val_binary, y_pred_val_binary)
    precision = precision_score(y_val_binary, y_pred_val_binary, zero_division=0)
    recall = recall_score(y_val_binary, y_pred_val_binary, zero_division=0)
    f1 = f1_score(y_val_binary, y_pred_val_binary, zero_division=0)

    results_val.append({
        'K': k,
        'threshold_val_pred': threshold_val_pred,
        'threshold_val_actual': threshold_val_actual,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    })

results_df_val = pd.DataFrame(results_val)
print("\n" + "="*60)
print("SUMMARY OF ALL K VALUES - VALIDATION")
print("="*60)
print(results_df_val.round(4))

# Best k
best_f1_idx_val = results_df_val['F1 Score'].idxmax()
best_row_val = results_df_val.loc[best_f1_idx_val]

print("\n" + "="*60)
print("BEST K BASED ON F1 SCORE - VALIDATION")
print("="*60)
print(best_row_val)

# Apply best k
y_val_binary_actual = convert_to_binary(y_val, best_row_val['threshold_val_actual'])
y_val_binary_pred = convert_to_binary(y_pred_val, best_row_val['threshold_val_pred'])

# Plot anomalies
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title(f"Validation Phase: Anomaly Detection (k={best_row_val['K']})")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")
ax1.plot(val_months, y_val, label="Validation Actual", color="pink", marker="o", alpha=0.6)
ax1.scatter(val_months[y_val_binary_actual == 1], y_val[y_val_binary_actual == 1],
            color="black", marker="x", s=100, label="Actual Anomaly")
ax1.axhline(best_row_val['threshold_val_actual'], color='pink', linestyle='--', label='Actual Threshold')
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(val_months, y_pred_val, label="Validation Predicted", color="purple", linestyle="--", marker="x", alpha=0.9)
ax2.scatter(val_months[y_val_binary_pred == 1], y_pred_val[y_val_binary_pred == 1],
            color="blue", marker="x", s=100, label="Predicted Anomaly")
ax2.axhline(best_row_val['threshold_val_pred'], color='purple', linestyle='--', label='Predicted Threshold')

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")
plt.show()

# Confusion Matrix
cm_val = confusion_matrix(y_val_binary_actual, y_val_binary_pred)
print("\nConfusion Matrix (Validation Set, Best k):")
print(cm_val)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_val, display_labels=["Normal", "Anomaly"])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix - Validation Set (Best k={best_row_val['K']})")
plt.show()

# ==== SHIFTED PREDICTIONS (VALIDATION) ====
print("=== AFTER SHIFT (VALIDATION) ===")
shift_results_val = []

for s in shift_range:
    shifted_predict_val = np.roll(y_pred_val, s)
    for k in k_values:
        threshold_val_pred = shifted_predict_val.mean() + (k * shifted_predict_val.std())
        threshold_val_actual = y_val.mean() + (k * y_val.std())

        y_val_binary = convert_to_binary(y_val, threshold_val_actual)
        y_pred_val_binary = convert_to_binary(shifted_predict_val, threshold_val_pred)

        binary_df = pd.DataFrame({
            'Actual': y_val_binary.flatten(),
            'Predicted': y_pred_val_binary.flatten()
        })
        '''filename = f'../outputs/GRU/anomaly_detection_val_shift_{s}_k_{k:.1f}_web_search_GRU.csv'
        binary_df.to_csv(filename, index=False)
        print(f"Saved binary data to: {filename}")'''

        accuracy = accuracy_score(y_val_binary, y_pred_val_binary)
        precision = precision_score(y_val_binary, y_pred_val_binary, zero_division=0)
        recall = recall_score(y_val_binary, y_pred_val_binary, zero_division=0)
        f1 = f1_score(y_val_binary, y_pred_val_binary, zero_division=0)

        shift_results_val.append({
            'Shift': s,
            'K': k,
            'threshold_val_pred': threshold_val_pred,
            'threshold_val_actual': threshold_val_actual,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1
        })

shift_results_df_val = pd.DataFrame(shift_results_val)
print("\n" + "="*60)
print("SUMMARY OF ALL SHIFT VALUES AND K VALUES - VALIDATION")
print("="*60)
print(shift_results_df_val.round(4))

best_f1_idx_shift_val = shift_results_df_val['F1 Score'].idxmax()
best_row_shift_val = shift_results_df_val.loc[best_f1_idx_shift_val]

print("\n" + "="*60)
print("BEST SHIFT BASED ON F1 SCORE - VALIDATION")
print("="*60)
print(best_row_shift_val)

# Plot & Confusion Matrix for Best Shift + K
best_shift_val = int(best_row_shift_val['Shift'])
best_k_val = best_row_shift_val['K']
shifted_predict_val = np.roll(y_pred_val, best_shift_val)

threshold_val_pred = shifted_predict_val.mean() + (best_k_val * shifted_predict_val.std())
threshold_val_actual = y_val.mean() + (best_k_val * y_val.std())

y_val_binary_actual = convert_to_binary(y_val, threshold_val_actual)
y_val_binary_pred = convert_to_binary(shifted_predict_val, threshold_val_pred)

fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title(f"Validation Phase (After Shift): Anomaly Detection (Shift={best_shift_val}, k={best_k_val})")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")
ax1.plot(val_months, y_val, label="Validation Actual", color="pink", marker="o", alpha=0.6)
ax1.scatter(val_months[y_val_binary_actual == 1], y_val[y_val_binary_actual == 1],
            color="black", marker="x", s=100, label="Actual Anomaly")
ax1.axhline(threshold_val_actual, color='pink', linestyle='--', label='Actual Threshold')
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(val_months, shifted_predict_val, label="Validation Predicted (Shifted)", color="purple", linestyle="--", marker="x", alpha=0.9)
ax2.scatter(val_months[y_val_binary_pred == 1], shifted_predict_val[y_val_binary_pred == 1],
            color="blue", marker="x", s=100, label="Predicted Anomaly")
ax2.axhline(threshold_val_pred, color='purple', linestyle='--', label='Predicted Threshold')

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")
plt.show()

cm_shift_val = confusion_matrix(y_val_binary_actual, y_val_binary_pred)
print("\nConfusion Matrix (Validation Set, Best Shift and K):")
print(cm_shift_val)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_shift_val, display_labels=["Normal", "Anomaly"])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix - Validation Set (Shift={best_shift_val}, k={best_k_val})")
plt.show()

# Compare with original (no shift) results
print("\n" + "="*60)
print("COMPARISON: NO SHIFT vs BEST SHIFT")
print("="*60)
print(f"No Shift - Best K: {best_row_val['K']}, F1 Score: {best_row_val['F1 Score']:.4f}")
print(f"With Shift - Best Shift: {best_shift_val}, Best K: {best_k_val}, F1 Score: {best_row_shift_val['F1 Score']:.4f}")
print(f"F1 Score Improvement: {best_row_shift_val['F1 Score'] - best_row_val['F1 Score']:.4f}")
# TEST ANOMALY DETECTION


print("\n" + "="*70)
print("ANOMALY DETECTION - TEST SET")
print("="*70)

k_values = [0.5, 0.7, 0.9, 1]
results_test = []

for k_pred in k_values:
    threshold_test_pred = (y_pred_test.mean() + (k_pred * y_pred_test.std()))
    for k_actual in k_values:
      threshold_test_actual = (y_test.mean() + (k_actual * y_test.std()))

      y_test_binary = convert_to_binary(y_test, threshold_test_actual)
      y_pred_test_binary = convert_to_binary(y_pred_test, threshold_test_pred)

      binary_df = pd.DataFrame({
          'Actual': y_test_binary.flatten(),
          'Predicted': y_pred_test_binary.flatten()
      })
      '''filename = f'../outputs/GRU/anomaly_detection_test_k_{k:.1f}_web_search_GRU.csv'
      binary_df.to_csv(filename, index=False)
      print(f"Saved binary data to: {filename}")'''

      accuracy = accuracy_score(y_test_binary, y_pred_test_binary)
      precision = precision_score(y_test_binary, y_pred_test_binary, zero_division=0)
      recall = recall_score(y_test_binary, y_pred_test_binary, zero_division=0)
      f1 = f1_score(y_test_binary, y_pred_test_binary, zero_division=0)

      results_test.append({
          'K_pred': k_pred,
          'K_actual': k_actual,
          'threshold_test_pred': threshold_test_pred,
          'threshold_test_actual': threshold_test_actual,
          'Accuracy': accuracy,
          'Precision': precision,
          'Recall': recall,
          'F1 Score': f1
      })

results_df_test = pd.DataFrame(results_test)
print("\n" + "="*60)
print("SUMMARY OF ALL K VALUES - TEST")
print("="*60)
print(results_df_test.round(4))

best_f1_idx_test = results_df_test['F1 Score'].idxmax()
best_row_test = results_df_test.loc[best_f1_idx_test]

print("\n" + "="*60)
print("BEST K BASED ON F1 SCORE - TEST")
print("="*60)
print(best_row_test)

# Apply best k
y_test_binary_actual = convert_to_binary(y_test, best_row_test['threshold_test_actual'])
y_test_binary_pred = convert_to_binary(y_pred_test, best_row_test['threshold_test_pred'])

# Plot anomalies
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title(f"Test Phase: Anomaly Detection (k_actual={best_row_test['K_actual']}, k_pred={best_row_test['K_pred']})")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")
ax1.plot(test_months, y_test, label="Test Actual", color="lightblue", marker="o", alpha=0.6)
ax1.scatter(test_months[y_test_binary_actual == 1], y_test[y_test_binary_actual == 1],
            color="black", marker="x", s=100, label="Actual Anomaly")
ax1.axhline(best_row_test['threshold_test_actual'], color='lightblue', linestyle='--', label='Actual Threshold')
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(test_months, y_pred_test, label="Test Predicted", color="blue", linestyle="--", marker="x", alpha=0.9)
ax2.scatter(test_months[y_test_binary_pred == 1], y_pred_test[y_test_binary_pred == 1],
            color="red", marker="x", s=100, label="Predicted Anomaly")
ax2.axhline(best_row_test['threshold_test_pred'], color='blue', linestyle='--', label='Predicted Threshold')

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")
plt.show()

# Confusion Matrix
cm_test = confusion_matrix(y_test_binary_actual, y_test_binary_pred)
print("\nConfusion Matrix (Test Set, Best k):")
print(cm_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=["Normal", "Anomaly"])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix - Test Set (Best k_actual={best_row_test['K_actual']}, k_pred={best_row_test['K_pred']})")
plt.show()

# ==== SHIFTED PREDICTIONS (TEST) ====
print("=== AFTER SHIFT (TEST) ===")
shift_results_test = []

for s in shift_range:
    shifted_predict_test = np.roll(y_pred_test, s)
    for k_pred in k_values:
      threshold_test_pred = shifted_predict_test.mean() + (k_pred * shifted_predict_test.std())
      for k_actual in k_values:
        threshold_test_actual = y_test.mean() + (k_actual * y_test.std())

        y_test_binary = convert_to_binary(y_test, threshold_test_actual)
        y_pred_test_binary = convert_to_binary(shifted_predict_test, threshold_test_pred)

        binary_df = pd.DataFrame({
            'Actual': y_test_binary.flatten(),
            'Predicted': y_pred_test_binary.flatten()
        })
        '''filename = f'../outputs/GRU/anomaly_detection_test_shift_{s}_k_{k:.1f}_web_search_GRU.csv'
        binary_df.to_csv(filename, index=False)
        print(f"Saved binary data to: {filename}")'''

        accuracy = accuracy_score(y_test_binary, y_pred_test_binary)
        precision = precision_score(y_test_binary, y_pred_test_binary, zero_division=0)
        recall = recall_score(y_test_binary, y_pred_test_binary, zero_division=0)
        f1 = f1_score(y_test_binary, y_pred_test_binary, zero_division=0)

        shift_results_test.append({
            'Shift': s,
            'K_pred': k_pred,
            'K_actual': k_actual,
            'threshold_test_pred': threshold_test_pred,
            'threshold_test_actual': threshold_test_actual,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1
        })

shift_results_df_test = pd.DataFrame(shift_results_test)
print("\n" + "="*60)
print("SUMMARY OF ALL SHIFT VALUES AND K VALUES - TEST")
print("="*60)
print(shift_results_df_test.round(4))

best_f1_idx_shift_test = shift_results_df_test['F1 Score'].idxmax()
best_row_shift_test = shift_results_df_test.loc[best_f1_idx_shift_test]

print("\n" + "="*60)
print("BEST SHIFT BASED ON F1 SCORE - TEST")
print("="*60)
print(best_row_shift_test)

# Plot & Confusion Matrix for Best Shift + K
best_shift_test = int(best_row_shift_test['Shift'])
best_k_actual_test = best_row_shift_test['K_actual']
best_k_pred_test = best_row_shift_test['K_pred']
shifted_predict_test = np.roll(y_pred_test, best_shift_test)

threshold_test_pred = shifted_predict_test.mean() + (best_k_pred_test * shifted_predict_test.std())
threshold_test_actual = y_test.mean() + (best_k_actual_test * y_test.std())

y_test_binary_actual = convert_to_binary(y_test, threshold_test_actual)
y_test_binary_pred = convert_to_binary(shifted_predict_test, threshold_test_pred)

fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title(f"Test Phase (After Shift): Anomaly Detection (Shift={best_shift_test}, k_actual={best_k_actual_test}, k_pred={best_k_pred_test})")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")
ax1.plot(test_months, y_test, label="Test Actual", color="lightblue", marker="o", alpha=0.6)
ax1.scatter(test_months[y_test_binary_actual == 1], y_test[y_test_binary_actual == 1],
            color="black", marker="x", s=100, label="Actual Anomaly")
ax1.axhline(threshold_test_actual, color='lightblue', linestyle='--', label='Actual Threshold')
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(test_months, shifted_predict_test, label="Test Predicted (Shifted)", color="blue", linestyle="--", marker="x", alpha=0.9)
ax2.scatter(test_months[y_test_binary_pred == 1], shifted_predict_test[y_test_binary_pred == 1],
            color="red", marker="x", s=100, label="Predicted Anomaly")
ax2.axhline(threshold_test_pred, color='blue', linestyle='--', label='Predicted Threshold')

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")
plt.show()

cm_shift_test = confusion_matrix(y_test_binary_actual, y_test_binary_pred)
print("\nConfusion Matrix (Test Set, Best Shift and K):")
print(cm_shift_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_shift_test, display_labels=["Normal", "Anomaly"])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix - Test Set (Shift={best_shift_test}, k_actual={best_k_actual_test}, k_pred={best_k_pred_test})")
plt.show()

# Compare with original (no shift) results
print("\n" + "="*60)
print("COMPARISON: NO SHIFT vs BEST SHIFT")
print("="*60)
print(f"No Shift - Best K_actual: {best_row_test['K_actual']}, k_pred={best_row_test['K_pred']} F1 Score: {best_row_test['F1 Score']:.4f}")
print(f"With Shift - Best Shift: {best_shift_test}, Best K_actual: {best_k_actual_test}, k_pred={best_k_pred_test}, F1 Score: {best_row_shift_test['F1 Score']:.4f}")
print(f"F1 Score Improvement: {best_row_shift_test['F1 Score'] - best_row_test['F1 Score']:.4f}")

print("\n" + "="*70)
print("ANOMALY DETECTION - TEST SET (USING BEST k FROM TRAINING)")
print("="*70)

# ==== CORRELATION ANALYSIS ====

print("\n" + "="*80)
print("CORRELATION BETWEEN ACTUAL AND PREDICTED VALUES")
print("="*80)

# Calculate Pearson correlation for each set
corr_train, p_value_train = pearsonr(y_train, y_pred_train)
corr_val, p_value_val = pearsonr(y_val, y_pred_val)
corr_test, p_value_test = pearsonr(y_test, y_pred_test)

print(f"Training Set Correlation: {corr_train:.4f} (p-value: {p_value_train:.4f})")
print(f"Validation Set Correlation: {corr_val:.4f} (p-value: {p_value_val:.4f})")
print(f"Test Set Correlation: {corr_test:.4f} (p-value: {p_value_test:.4f})")

# Create correlation summary
correlation_summary = pd.DataFrame({
    'Dataset': ['Training', 'Validation', 'Test'],
    'Correlation': [corr_train, corr_val, corr_test],
    'P-value': [p_value_train, p_value_val, p_value_test]
})

print("\nCorrelation Summary:")
print(correlation_summary.round(4))

# Plot correlation scatter plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training set
axes[0].scatter(y_train, y_pred_train, alpha=0.6)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Training Set\nCorrelation: {corr_train:.4f}')
axes[0].grid(True)

# Validation set
axes[1].scatter(y_val, y_pred_val, alpha=0.6)
axes[1].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title(f'Validation Set\nCorrelation: {corr_val:.4f}')
axes[1].grid(True)

# Test set
axes[2].scatter(y_test, y_pred_test, alpha=0.6)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[2].set_xlabel('Actual')
axes[2].set_ylabel('Predicted')
axes[2].set_title(f'Test Set\nCorrelation: {corr_test:.4f}')
axes[2].grid(True)

plt.tight_layout()
plt.show()

print(f"\nCORRELATION SUMMARY:")
print(f"Training Set Correlation: {corr_train:.4f}")
print(f"Validation Set Correlation: {corr_val:.4f}")
print(f"Test Set Correlation: {corr_test:.4f}")

# Youtube Dataset

In [19]:
dataset='../Datasets/dataset_youtube_eradah.csv'

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, GRU, Input, LayerNormalization
from tensorflow.keras.optimizers import Adam
import math
from scipy.stats import pearsonr

# Function to calculate MAPE
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Function to calculate sMAPE
def symmetric_mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))



# Read the dataset
df = pd.read_csv(dataset)

# Convert all columns to numeric
for col in df.columns[1:]:
    df[col] = pd.to_numeric(df[col])

# Convert month to datetime and set as index
df['month'] = pd.to_datetime(df['month'])
df.set_index('month', inplace=True)


# Display dataset info
print("Dataset shape:", df.shape)
print("\nDataset info:")
print(df.info())
print("\nFirst few rows:")
print(df.head())

# Separate features and target
feature_columns = ['depression', 'tired', 'weight loss', 'anxiety', 'sad']
target_column = ['actual']

X = df[feature_columns]
y = df[target_column]

# Display original ranges
print("\nOriginal feature ranges:")
print(X.agg(['min', 'max']))
print("\nOriginal target range:")
print(y.agg(['min', 'max']))

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

# Scale features and target separately
X_scaled = feature_scaler.fit_transform(X)
y_scaled = target_scaler.fit_transform(y)

# Create scaled DataFrames
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_columns, index=df.index)
y_scaled_df = pd.DataFrame(y_scaled, columns=target_column, index=df.index)

# Combine
scaled_df = pd.concat([X_scaled_df, y_scaled_df], axis=1)

print("\nAfter scaling - Feature ranges (0-1):")
print(X_scaled_df.agg(['min', 'max']))
print("\nAfter scaling - Target range (0-1):")
print(y_scaled_df.agg(['min', 'max']))


scaled_data = np.column_stack([X_scaled, y_scaled])


# Create sequences
def create_sequences(data, time_steps):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:(i + time_steps), :-1])  # All features except the target
        y.append(data[i + time_steps, -1])       # Target (actual cases)
    return np.array(X), np.array(y)

# Set time steps and create sequences
time_steps = 12
X, y = create_sequences(scaled_data, time_steps)

print(f"\nShape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

# Split the data: 55% training, 20% validation, 25% test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.25, random_state=42, shuffle=False)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.267, random_state=42, shuffle=False)

print(f"Training set: {X_train.shape}, {y_train.shape}")
print(f"Validation set: {X_val.shape}, {y_val.shape}")
print(f"Test set: {X_test.shape}, {y_test.shape}")

# Build GRU model
model = Sequential([
    Input(shape=(X_train.shape[1], X_train.shape[2])),
    Bidirectional(GRU(128, return_sequences=True)),
    LayerNormalization(),
    Bidirectional(GRU(64, return_sequences=False)),
    LayerNormalization(),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1)
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=40, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("best_model.keras", save_best_only=True)
]
# Display model summary
print("\nModel Summary:")
model.summary()

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=8,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1,
    shuffle=False
)

# Make predictions
y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

# Calculate evaluation metrics
def calculate_metrics(y_true, y_pred, set_name):
    mse = mean_squared_error(y_true, y_pred)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    smape = symmetric_mean_absolute_percentage_error(y_true, y_pred)

    print(f"\n{set_name} Set Metrics:")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MAPE: {mape:.2f}%")
    print(f"sMAPE: {smape:.2f}%")

    return mse, rmse, mae, mape, smape

# Calculate metrics for all sets
train_metrics = calculate_metrics(y_train, y_pred_train, "Training")
val_metrics = calculate_metrics(y_val, y_pred_val, "Validation")
test_metrics = calculate_metrics(y_test, y_pred_test, "Test")

# Create a comparison DataFrame
metrics_df = pd.DataFrame({
    'Dataset': ['Training', 'Validation', 'Test'],
    'MSE': [train_metrics[0], val_metrics[0], test_metrics[0]],
    'RMSE': [train_metrics[1], val_metrics[1], test_metrics[1]],
    'MAE': [train_metrics[2], val_metrics[2], test_metrics[2]],
    'sMAPE': [train_metrics[4], val_metrics[4], test_metrics[4]]
})

print("\n" + "="*50)
print("COMPARISON OF ALL DATASET PERFORMANCE")
print("="*50)
print(metrics_df.round(4))

# Plot training history
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss During Training')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()



all_months = df.index[time_steps:]

train_months = all_months[:len(y_train)]
val_months = all_months[len(y_train):len(y_train) + len(y_val)]
test_months = all_months[-len(y_test):]

# Ensure 1-D arrays
y_train= np.array(y_train).reshape(-1)
y_pred_train = np.array(y_pred_train).reshape(-1)

y_val = np.array(y_val).reshape(-1)
y_pred_val = np.array(y_pred_val).reshape(-1)

y_test = np.array(y_test).reshape(-1)
y_pred_test = np.array(y_pred_test).reshape(-1)

# --- Train ---
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title("Train Phase: Actual vs Predicted Depression Cases")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")
ax1.plot(train_months, y_train, label="Train Actual", color="orange", marker="o", alpha=0.6)
ax1.tick_params(axis='y', labelcolor="black")
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(train_months, y_pred_train, label="Train Predicted", color="red", linestyle="--", marker="x", alpha=0.9)
ax2.tick_params(axis='y', labelcolor="black")

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left")

plt.show()

# --- Validation ---
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title("Validation Phase: Actual vs Predicted Depression Cases")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")
ax1.plot(val_months, y_val, label="Validation Actual", color="pink", marker="o", alpha=0.6)
ax1.tick_params(axis='y', labelcolor="black")
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(val_months, y_pred_val, label="Validation Predicted", color="purple", linestyle="--", marker="x", alpha=0.9)
ax2.tick_params(axis='y', labelcolor="black")

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left")

plt.show()

# --- Test ---
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title("Test Phase: Actual vs Predicted Depression Cases")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")
ax1.plot(test_months, y_test, label="Test Actual", color="lightblue", marker="o", alpha=0.6)
ax1.tick_params(axis='y', labelcolor="black")
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(test_months, y_pred_test, label="Test Predicted", color="blue", linestyle="--", marker="x", alpha=0.9)
ax2.tick_params(axis='y', labelcolor="black")

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left")

plt.show()

# ==== Anomaly Detection ====

# Convert to binary (1 = anomaly, 0 = normal)
def convert_to_binary(values, threshold):
    return (values > threshold).astype(int)

k_values = [0.5, 0.7, 0.9, 1, 1.3, 1.5, 1.7, 2]

# --- Train ---
results_train = []
for k_actual in k_values:
  threshold_train_actual = (y_train.mean() + (k_actual * y_train.std()))
  for k_pred in k_values:
    threshold_train_pred = (y_pred_train.mean() + (k_pred * y_pred_train.std()))

    y_train_binary = convert_to_binary(y_train, threshold_train_actual)
    y_pred_train_binary = convert_to_binary(y_pred_train, threshold_train_pred)

    binary_df = pd.DataFrame({
        'Actual': y_train_binary.flatten(),
        'Predicted': y_pred_train_binary.flatten()
    })

    # Save to CSV file
    '''filename = f'../outputs/GRU/anomaly_detection_train_kActual_{k_actual:.1f}_kPred_{k_pred:.1f}_yt_search_GRU.csv'
    binary_df.to_csv(filename, index=False)
    print(f"Saved binary data to: {filename}")'''

    accuracy = accuracy_score(y_train_binary, y_pred_train_binary)
    precision = precision_score(y_train_binary, y_pred_train_binary, zero_division=0)
    recall = recall_score(y_train_binary, y_pred_train_binary, zero_division=0)
    f1 = f1_score(y_train_binary, y_pred_train_binary, zero_division=0)

    results_train.append({
        'K_actual': k_actual,
        'K_pred': k_pred,
        'threshold_train_pred': threshold_train_pred,
        'threshold_train_actual': threshold_train_actual,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    })

results_df_train = pd.DataFrame(results_train)
print("\n" + "="*60)
print("SUMMARY OF ALL K VALUES - TRAINING")
print("="*60)
print(results_df_train.round(4))

# Best k
best_f1_idx_train = results_df_train['F1 Score'].idxmax()
best_row_train = results_df_train.loc[best_f1_idx_train]

print("\n" + "="*60)
print("BEST K BASED ON F1 SCORE - TRAINING")
print("="*60)
print(best_row_train)

# Apply best k
y_train_binary_actual = convert_to_binary(y_train, best_row_train['threshold_train_actual'])
y_train_binary_pred = convert_to_binary(y_pred_train, best_row_train['threshold_train_pred'])

train_months = np.array(train_months)
y_pred_train = y_pred_train.flatten()
# Plot anomalies
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title(f"Train Phase: Anomaly Detection (k_actual={best_row_train['K_actual']}, k_pred={best_row_train['K_pred']})")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")
ax1.plot(train_months, y_train, label="Train Actual", color="orange", marker="o", alpha=0.6)
ax1.scatter(train_months[y_train_binary_actual == 1], y_train[y_train_binary_actual == 1],
            color="black", marker="x", s=100, label="Actual Anomaly")

# Add threshold line for actual values
ax1.axhline(best_row_train['threshold_train_actual'], color='orange', linestyle='--', label='Actual Threshold')

ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(train_months, y_pred_train, label="Train Predicted", color="red", linestyle="--", marker="x", alpha=0.9)
ax2.scatter(train_months[y_train_binary_pred == 1], y_pred_train[y_train_binary_pred == 1],
            color="blue", marker="x", s=100, label="Predicted Anomaly")

# Add threshold line for predicted values
ax2.axhline(best_row_train['threshold_train_pred'], color='red', linestyle='--', label='Predicted Threshold')

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

plt.show()

# Confusion matrix
cm_train = confusion_matrix(y_train_binary_actual, y_train_binary_pred)
print("\nConfusion Matrix (Training Set):")
print(cm_train)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_train, display_labels=["Normal", "Anomaly"])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix - Training Set (Best k_actual={best_row_train['K_actual']}, k_pred={best_row_train['K_pred']})")
plt.show()

print("=== AFTER APPLYING SHIFT ===")

shift_results = []
shift_range = np.arange(1, 4)

for s in shift_range:
    # Apply shift
    shifted_predict_train = np.roll(y_pred_train, s)

    for k_actual in k_values:
      threshold_train_actual = (y_train.mean() + (k_actual * y_train.std()))
      for k_pred in k_values:
        threshold_train_pred = (shifted_predict_train.mean() + (k_pred * shifted_predict_train.std()))

        y_train_binary = convert_to_binary(y_train, threshold_train_actual)
        y_pred_train_binary = convert_to_binary(shifted_predict_train, threshold_train_pred)

        binary_df = pd.DataFrame({
            'Actual': y_train_binary.flatten(),
            'Predicted': y_pred_train_binary.flatten()
        })
        '''filename = f'../outputs/GRU/anomaly_detection_train_shift_{s}_kActual_{k_actual:.1f}_kPred_{k_pred:.1f}_yt_search_GRU.csv'
        binary_df.to_csv(filename, index=False)
        print(f"Saved binary data to: {filename}")'''

        accuracy = accuracy_score(y_train_binary, y_pred_train_binary)
        precision = precision_score(y_train_binary, y_pred_train_binary, zero_division=0)
        recall = recall_score(y_train_binary, y_pred_train_binary, zero_division=0)
        f1 = f1_score(y_train_binary, y_pred_train_binary, zero_division=0)

        shift_results.append({
            'Shift': s,
            'K_actual': k_actual,
            'K_pred': k_pred,
            'threshold_train_pred': threshold_train_pred,
            'threshold_train_actual': threshold_train_actual,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1
        })

shift_results_df = pd.DataFrame(shift_results)
print("\n" + "="*60)
print("SUMMARY OF ALL SHIFT VALUES AND K VALUES - TRAINING")
print("="*60)
print(shift_results_df.round(4))

best_f1_idx_shift_train = shift_results_df['F1 Score'].idxmax()
best_row_shift_train = shift_results_df.loc[best_f1_idx_shift_train]

print("\n" + "="*60)
print("BEST SHIFT BASED ON F1 SCORE - TRAINING")
print("="*60)
print(best_row_shift_train)

# === PLOT & CONFUSION MATRIX FOR BEST SHIFT AND K ===

best_shift = int(best_row_shift_train['Shift'])
best_k_actual = best_row_shift_train['K_actual']
best_k_pred = best_row_shift_train['K_pred']

# Apply shift
shifted_predict_train = np.roll(y_pred_train, best_shift)

threshold_train_pred = shifted_predict_train.mean() + (best_k_pred * shifted_predict_train.std())
threshold_train_actual = y_train.mean() + (best_k_actual * y_train.std())

y_train_binary_actual = convert_to_binary(y_train, threshold_train_actual)
y_train_binary_pred = convert_to_binary(shifted_predict_train, threshold_train_pred)

# --- PLOT ---
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title(f"Train Phase (After Shift): Anomaly Detection (Shift={best_shift}, k_actual={best_k_actual}, k_pred={best_k_pred})")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")

ax1.plot(train_months, y_train, label="Train Actual", color="orange", marker="o", alpha=0.6)
ax1.scatter(train_months[y_train_binary_actual == 1], y_train[y_train_binary_actual == 1],
            color="black", marker="x", s=100, label="Actual Anomaly")
ax1.axhline(threshold_train_actual, color='orange', linestyle='--', label='Actual Threshold')
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(train_months, shifted_predict_train, label="Train Predicted (Shifted)", color="red", linestyle="--", marker="x", alpha=0.9)
ax2.scatter(train_months[y_train_binary_pred == 1], shifted_predict_train[y_train_binary_pred == 1],
            color="blue", marker="x", s=100, label="Predicted Anomaly")
ax2.axhline(threshold_train_pred, color='red', linestyle='--', label='Predicted Threshold')

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

plt.show()

# --- CONFUSION MATRIX ---
cm_shift_train = confusion_matrix(y_train_binary_actual, y_train_binary_pred)
print("\nConfusion Matrix (Training Set):")
print(cm_shift_train)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_shift_train, display_labels=["Normal", "Anomaly"])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix - Training Set (Shift={best_shift}, k_actual={best_k_actual}, k_pred={best_k_pred})")
plt.show()

# Compare with original (no shift) results
print("\n" + "="*60)
print("COMPARISON: NO SHIFT vs BEST SHIFT")
print("="*60)
print(f"No Shift - Best K_actual: {best_row_train['K_actual']}, K_pred: {best_row_train['K_pred']}, F1 Score: {best_row_train['F1 Score']:.4f}")
print(f"With Shift - Best Shift: {best_shift}, Best K_actual: {best_k_actual}, K_pred: {best_k_pred}, F1 Score: {best_row_shift_train['F1 Score']:.4f}")
print(f"F1 Score Improvement: {best_row_shift_train['F1 Score'] - best_row_train['F1 Score']:.4f}")

results_val = []

for k in k_values:
    threshold_val_pred = (y_pred_val.mean() + (k * y_pred_val.std()))
    threshold_val_actual = (y_val.mean() + (k * y_val.std()))

    y_val_binary = convert_to_binary(y_val, threshold_val_actual)
    y_pred_val_binary = convert_to_binary(y_pred_val, threshold_val_pred)

    binary_df = pd.DataFrame({
        'Actual': y_val_binary.flatten(),
        'Predicted': y_pred_val_binary.flatten()
    })
    '''filename = f'../outputs/GRU/anomaly_detection_val_k_{k:.1f}_yt_search_GRU.csv'
    binary_df.to_csv(filename, index=False)
    print(f"Saved binary data to: {filename}")'''

    accuracy = accuracy_score(y_val_binary, y_pred_val_binary)
    precision = precision_score(y_val_binary, y_pred_val_binary, zero_division=0)
    recall = recall_score(y_val_binary, y_pred_val_binary, zero_division=0)
    f1 = f1_score(y_val_binary, y_pred_val_binary, zero_division=0)

    results_val.append({
        'K': k,
        'threshold_val_pred': threshold_val_pred,
        'threshold_val_actual': threshold_val_actual,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    })

results_df_val = pd.DataFrame(results_val)
print("\n" + "="*60)
print("SUMMARY OF ALL K VALUES - VALIDATION")
print("="*60)
print(results_df_val.round(4))

# Best k
best_f1_idx_val = results_df_val['F1 Score'].idxmax()
best_row_val = results_df_val.loc[best_f1_idx_val]

print("\n" + "="*60)
print("BEST K BASED ON F1 SCORE - VALIDATION")
print("="*60)
print(best_row_val)

# Apply best k
y_val_binary_actual = convert_to_binary(y_val, best_row_val['threshold_val_actual'])
y_val_binary_pred = convert_to_binary(y_pred_val, best_row_val['threshold_val_pred'])

# Plot anomalies
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title(f"Validation Phase: Anomaly Detection (k={best_row_val['K']})")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")
ax1.plot(val_months, y_val, label="Validation Actual", color="pink", marker="o", alpha=0.6)
ax1.scatter(val_months[y_val_binary_actual == 1], y_val[y_val_binary_actual == 1],
            color="black", marker="x", s=100, label="Actual Anomaly")
ax1.axhline(best_row_val['threshold_val_actual'], color='pink', linestyle='--', label='Actual Threshold')
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(val_months, y_pred_val, label="Validation Predicted", color="purple", linestyle="--", marker="x", alpha=0.9)
ax2.scatter(val_months[y_val_binary_pred == 1], y_pred_val[y_val_binary_pred == 1],
            color="blue", marker="x", s=100, label="Predicted Anomaly")
ax2.axhline(best_row_val['threshold_val_pred'], color='purple', linestyle='--', label='Predicted Threshold')

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")
plt.show()

# Confusion Matrix
cm_val = confusion_matrix(y_val_binary_actual, y_val_binary_pred)
print("\nConfusion Matrix (Validation Set, Best k):")
print(cm_val)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_val, display_labels=["Normal", "Anomaly"])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix - Validation Set (Best k={best_row_val['K']})")
plt.show()

# ==== SHIFTED PREDICTIONS (VALIDATION) ====
print("=== AFTER SHIFT (VALIDATION) ===")
shift_results_val = []

for s in shift_range:
    shifted_predict_val = np.roll(y_pred_val, s)
    for k in k_values:
        threshold_val_pred = shifted_predict_val.mean() + (k * shifted_predict_val.std())
        threshold_val_actual = y_val.mean() + (k * y_val.std())

        y_val_binary = convert_to_binary(y_val, threshold_val_actual)
        y_pred_val_binary = convert_to_binary(shifted_predict_val, threshold_val_pred)

        binary_df = pd.DataFrame({
            'Actual': y_val_binary.flatten(),
            'Predicted': y_pred_val_binary.flatten()
        })
        '''filename = f'../outputs/GRU/anomaly_detection_val_shift_{s}_k_{k:.1f}_yt_search_GRU.csv'
        binary_df.to_csv(filename, index=False)
        print(f"Saved binary data to: {filename}")'''

        accuracy = accuracy_score(y_val_binary, y_pred_val_binary)
        precision = precision_score(y_val_binary, y_pred_val_binary, zero_division=0)
        recall = recall_score(y_val_binary, y_pred_val_binary, zero_division=0)
        f1 = f1_score(y_val_binary, y_pred_val_binary, zero_division=0)

        shift_results_val.append({
            'Shift': s,
            'K': k,
            'threshold_val_pred': threshold_val_pred,
            'threshold_val_actual': threshold_val_actual,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1
        })

shift_results_df_val = pd.DataFrame(shift_results_val)
print("\n" + "="*60)
print("SUMMARY OF ALL SHIFT VALUES AND K VALUES - VALIDATION")
print("="*60)
print(shift_results_df_val.round(4))

best_f1_idx_shift_val = shift_results_df_val['F1 Score'].idxmax()
best_row_shift_val = shift_results_df_val.loc[best_f1_idx_shift_val]

print("\n" + "="*60)
print("BEST SHIFT BASED ON F1 SCORE - VALIDATION")
print("="*60)
print(best_row_shift_val)

# Plot & Confusion Matrix for Best Shift + K
best_shift_val = int(best_row_shift_val['Shift'])
best_k_val = best_row_shift_val['K']
shifted_predict_val = np.roll(y_pred_val, best_shift_val)

threshold_val_pred = shifted_predict_val.mean() + (best_k_val * shifted_predict_val.std())
threshold_val_actual = y_val.mean() + (best_k_val * y_val.std())

y_val_binary_actual = convert_to_binary(y_val, threshold_val_actual)
y_val_binary_pred = convert_to_binary(shifted_predict_val, threshold_val_pred)

fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title(f"Validation Phase (After Shift): Anomaly Detection (Shift={best_shift_val}, k={best_k_val})")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")
ax1.plot(val_months, y_val, label="Validation Actual", color="pink", marker="o", alpha=0.6)
ax1.scatter(val_months[y_val_binary_actual == 1], y_val[y_val_binary_actual == 1],
            color="black", marker="x", s=100, label="Actual Anomaly")
ax1.axhline(threshold_val_actual, color='pink', linestyle='--', label='Actual Threshold')
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(val_months, shifted_predict_val, label="Validation Predicted (Shifted)", color="purple", linestyle="--", marker="x", alpha=0.9)
ax2.scatter(val_months[y_val_binary_pred == 1], shifted_predict_val[y_val_binary_pred == 1],
            color="blue", marker="x", s=100, label="Predicted Anomaly")
ax2.axhline(threshold_val_pred, color='purple', linestyle='--', label='Predicted Threshold')

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")
plt.show()

cm_shift_val = confusion_matrix(y_val_binary_actual, y_val_binary_pred)
print("\nConfusion Matrix (Validation Set, Best Shift and K):")
print(cm_shift_val)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_shift_val, display_labels=["Normal", "Anomaly"])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix - Validation Set (Shift={best_shift_val}, k={best_k_val})")
plt.show()

# Compare with original (no shift) results
print("\n" + "="*60)
print("COMPARISON: NO SHIFT vs BEST SHIFT")
print("="*60)
print(f"No Shift - Best K: {best_row_val['K']}, F1 Score: {best_row_val['F1 Score']:.4f}")
print(f"With Shift - Best Shift: {best_shift_val}, Best K: {best_k_val}, F1 Score: {best_row_shift_val['F1 Score']:.4f}")
print(f"F1 Score Improvement: {best_row_shift_val['F1 Score'] - best_row_val['F1 Score']:.4f}")
# TEST ANOMALY DETECTION


print("\n" + "="*70)
print("ANOMALY DETECTION - TEST SET")
print("="*70)

k_values = [0.5, 0.7, 0.9, 1]
results_test = []

for k_pred in k_values:
    threshold_test_pred = (y_pred_test.mean() + (k_pred * y_pred_test.std()))
    for k_actual in k_values:
      threshold_test_actual = (y_test.mean() + (k_actual * y_test.std()))

      y_test_binary = convert_to_binary(y_test, threshold_test_actual)
      y_pred_test_binary = convert_to_binary(y_pred_test, threshold_test_pred)

      binary_df = pd.DataFrame({
          'Actual': y_test_binary.flatten(),
          'Predicted': y_pred_test_binary.flatten()
      })
      '''filename = f'../outputs/GRU/anomaly_detection_test_k_{k:.1f}_yt_search_GRU.csv'
      binary_df.to_csv(filename, index=False)
      print(f"Saved binary data to: {filename}")'''

      accuracy = accuracy_score(y_test_binary, y_pred_test_binary)
      precision = precision_score(y_test_binary, y_pred_test_binary, zero_division=0)
      recall = recall_score(y_test_binary, y_pred_test_binary, zero_division=0)
      f1 = f1_score(y_test_binary, y_pred_test_binary, zero_division=0)

      results_test.append({
          'K_pred': k_pred,
          'K_actual': k_actual,
          'threshold_test_pred': threshold_test_pred,
          'threshold_test_actual': threshold_test_actual,
          'Accuracy': accuracy,
          'Precision': precision,
          'Recall': recall,
          'F1 Score': f1
      })

results_df_test = pd.DataFrame(results_test)
print("\n" + "="*60)
print("SUMMARY OF ALL K VALUES - TEST")
print("="*60)
print(results_df_test.round(4))

best_f1_idx_test = results_df_test['F1 Score'].idxmax()
best_row_test = results_df_test.loc[best_f1_idx_test]

print("\n" + "="*60)
print("BEST K BASED ON F1 SCORE - TEST")
print("="*60)
print(best_row_test)

# Apply best k
y_test_binary_actual = convert_to_binary(y_test, best_row_test['threshold_test_actual'])
y_test_binary_pred = convert_to_binary(y_pred_test, best_row_test['threshold_test_pred'])

# Plot anomalies
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title(f"Test Phase: Anomaly Detection (k_actual={best_row_test['K_actual']}, k_pred={best_row_test['K_pred']})")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")
ax1.plot(test_months, y_test, label="Test Actual", color="lightblue", marker="o", alpha=0.6)
ax1.scatter(test_months[y_test_binary_actual == 1], y_test[y_test_binary_actual == 1],
            color="black", marker="x", s=100, label="Actual Anomaly")
ax1.axhline(best_row_test['threshold_test_actual'], color='lightblue', linestyle='--', label='Actual Threshold')
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(test_months, y_pred_test, label="Test Predicted", color="blue", linestyle="--", marker="x", alpha=0.9)
ax2.scatter(test_months[y_test_binary_pred == 1], y_pred_test[y_test_binary_pred == 1],
            color="red", marker="x", s=100, label="Predicted Anomaly")
ax2.axhline(best_row_test['threshold_test_pred'], color='blue', linestyle='--', label='Predicted Threshold')

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")
plt.show()

# Confusion Matrix
cm_test = confusion_matrix(y_test_binary_actual, y_test_binary_pred)
print("\nConfusion Matrix (Test Set, Best k):")
print(cm_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=["Normal", "Anomaly"])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix - Test Set (Best k_actual={best_row_test['K_actual']}, k_pred={best_row_test['K_pred']})")
plt.show()

# ==== SHIFTED PREDICTIONS (TEST) ====
print("=== AFTER SHIFT (TEST) ===")
shift_results_test = []

for s in shift_range:
    shifted_predict_test = np.roll(y_pred_test, s)
    for k_pred in k_values:
      threshold_test_pred = shifted_predict_test.mean() + (k_pred * shifted_predict_test.std())
      for k_actual in k_values:
        threshold_test_actual = y_test.mean() + (k_actual * y_test.std())

        y_test_binary = convert_to_binary(y_test, threshold_test_actual)
        y_pred_test_binary = convert_to_binary(shifted_predict_test, threshold_test_pred)

        binary_df = pd.DataFrame({
            'Actual': y_test_binary.flatten(),
            'Predicted': y_pred_test_binary.flatten()
        })
        '''filename = f'../outputs/GRU/anomaly_detection_test_shift_{s}_k_{k:.1f}_yt_search_GRU.csv'
        binary_df.to_csv(filename, index=False)
        print(f"Saved binary data to: {filename}")'''

        accuracy = accuracy_score(y_test_binary, y_pred_test_binary)
        precision = precision_score(y_test_binary, y_pred_test_binary, zero_division=0)
        recall = recall_score(y_test_binary, y_pred_test_binary, zero_division=0)
        f1 = f1_score(y_test_binary, y_pred_test_binary, zero_division=0)

        shift_results_test.append({
            'Shift': s,
            'K_pred': k_pred,
            'K_actual': k_actual,
            'threshold_test_pred': threshold_test_pred,
            'threshold_test_actual': threshold_test_actual,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1
        })

shift_results_df_test = pd.DataFrame(shift_results_test)
print("\n" + "="*60)
print("SUMMARY OF ALL SHIFT VALUES AND K VALUES - TEST")
print("="*60)
print(shift_results_df_test.round(4))

best_f1_idx_shift_test = shift_results_df_test['F1 Score'].idxmax()
best_row_shift_test = shift_results_df_test.loc[best_f1_idx_shift_test]

print("\n" + "="*60)
print("BEST SHIFT BASED ON F1 SCORE - TEST")
print("="*60)
print(best_row_shift_test)

# Plot & Confusion Matrix for Best Shift + K
best_shift_test = int(best_row_shift_test['Shift'])
best_k_actual_test = best_row_shift_test['K_actual']
best_k_pred_test = best_row_shift_test['K_pred']
shifted_predict_test = np.roll(y_pred_test, best_shift_test)

threshold_test_pred = shifted_predict_test.mean() + (best_k_pred_test * shifted_predict_test.std())
threshold_test_actual = y_test.mean() + (best_k_actual_test * y_test.std())

y_test_binary_actual = convert_to_binary(y_test, threshold_test_actual)
y_test_binary_pred = convert_to_binary(shifted_predict_test, threshold_test_pred)

fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.set_title(f"Test Phase (After Shift): Anomaly Detection (Shift={best_shift_test}, k_actual={best_k_actual_test}, k_pred={best_k_pred_test})")
ax1.set_xlabel("Month")
ax1.set_ylabel("Actual Depression Cases", color="black")
ax1.plot(test_months, y_test, label="Test Actual", color="lightblue", marker="o", alpha=0.6)
ax1.scatter(test_months[y_test_binary_actual == 1], y_test[y_test_binary_actual == 1],
            color="black", marker="x", s=100, label="Actual Anomaly")
ax1.axhline(threshold_test_actual, color='lightblue', linestyle='--', label='Actual Threshold')
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel("Predicted Depression Cases", color="black")
ax2.plot(test_months, shifted_predict_test, label="Test Predicted (Shifted)", color="blue", linestyle="--", marker="x", alpha=0.9)
ax2.scatter(test_months[y_test_binary_pred == 1], shifted_predict_test[y_test_binary_pred == 1],
            color="red", marker="x", s=100, label="Predicted Anomaly")
ax2.axhline(threshold_test_pred, color='blue', linestyle='--', label='Predicted Threshold')

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")
plt.show()

cm_shift_test = confusion_matrix(y_test_binary_actual, y_test_binary_pred)
print("\nConfusion Matrix (Test Set, Best Shift and K):")
print(cm_shift_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_shift_test, display_labels=["Normal", "Anomaly"])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix - Test Set (Shift={best_shift_test}, k_actual={best_k_actual_test}, k_pred={best_k_pred_test})")
plt.show()

# Compare with original (no shift) results
print("\n" + "="*60)
print("COMPARISON: NO SHIFT vs BEST SHIFT")
print("="*60)
print(f"No Shift - Best K_actual: {best_row_test['K_actual']}, k_pred={best_row_test['K_pred']} F1 Score: {best_row_test['F1 Score']:.4f}")
print(f"With Shift - Best Shift: {best_shift_test}, Best K_actual: {best_k_actual_test}, k_pred={best_k_pred_test}, F1 Score: {best_row_shift_test['F1 Score']:.4f}")
print(f"F1 Score Improvement: {best_row_shift_test['F1 Score'] - best_row_test['F1 Score']:.4f}")

print("\n" + "="*70)
print("ANOMALY DETECTION - TEST SET (USING BEST k FROM TRAINING)")
print("="*70)

# ==== CORRELATION ANALYSIS ====

print("\n" + "="*80)
print("CORRELATION BETWEEN ACTUAL AND PREDICTED VALUES")
print("="*80)

# Calculate Pearson correlation for each set
corr_train, p_value_train = pearsonr(y_train, y_pred_train)
corr_val, p_value_val = pearsonr(y_val, y_pred_val)
corr_test, p_value_test = pearsonr(y_test, y_pred_test)

print(f"Training Set Correlation: {corr_train:.4f} (p-value: {p_value_train:.4f})")
print(f"Validation Set Correlation: {corr_val:.4f} (p-value: {p_value_val:.4f})")
print(f"Test Set Correlation: {corr_test:.4f} (p-value: {p_value_test:.4f})")

# Create correlation summary
correlation_summary = pd.DataFrame({
    'Dataset': ['Training', 'Validation', 'Test'],
    'Correlation': [corr_train, corr_val, corr_test],
    'P-value': [p_value_train, p_value_val, p_value_test]
})

print("\nCorrelation Summary:")
print(correlation_summary.round(4))

# Plot correlation scatter plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training set
axes[0].scatter(y_train, y_pred_train, alpha=0.6)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Training Set\nCorrelation: {corr_train:.4f}')
axes[0].grid(True)

# Validation set
axes[1].scatter(y_val, y_pred_val, alpha=0.6)
axes[1].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title(f'Validation Set\nCorrelation: {corr_val:.4f}')
axes[1].grid(True)

# Test set
axes[2].scatter(y_test, y_pred_test, alpha=0.6)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[2].set_xlabel('Actual')
axes[2].set_ylabel('Predicted')
axes[2].set_title(f'Test Set\nCorrelation: {corr_test:.4f}')
axes[2].grid(True)

plt.tight_layout()
plt.show()

print(f"\nCORRELATION SUMMARY:")
print(f"Training Set Correlation: {corr_train:.4f}")
print(f"Validation Set Correlation: {corr_val:.4f}")
print(f"Test Set Correlation: {corr_test:.4f}")